In [17]:
import os
import requests
import time
from tqdm import tqdm
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langchain.tools import tool
from dotenv import load_dotenv
import json

# Configuration
load_dotenv()
LLM_MODEL = "gemini-2.5-flash-lite"
googel_api_key = os.getenv("GOOGLE_API_KEY")

# API Endpoints
URL_NET = "http://localhost:8000/network-awareness"
URL_LOC_UES = "http://localhost:8001/location/get-ues"
URL_LOC_GNBS = "http://localhost:8001/location/get-gnbs"
URL_ZONES = "http://localhost:8001/zones"

In [12]:
import requests
import json

def get_processed_network_state():
    print("--- Starting Data Preprocessing Test ---\n")

    try:
        # 1. Fetch data from your APIs
        # Note: Using a short timeout to fail fast if your servers are down
        net_map = requests.get(URL_NET, timeout=2).json().get('radio_map', [])
        ue_locs_list = requests.get(URL_LOC_UES, timeout=2).json()
        gnb_locs_list = requests.get(URL_LOC_GNBS, timeout=2).json()
        zones = requests.get(URL_ZONES, timeout=2).json()
        
        print("[Status] Successfully fetched live data from Localhost.")

    except Exception as e:
        print(f"[Status] Localhost APIs not reachable ({e}).")
        print("[Status] Falling back to provided Sample Data for structure verification...")
        
        # MOCK DATA (from your prompt) for testing if servers are off
        net_map = [
            {"gnb_id": "00000001", "connected_ues": [{"supi": "imsi-208930000000002"}]},
            {"gnb_id": "00000003", "connected_ues": [{"supi": "imsi-208930000000001"}]}
        ]
        ue_locs_list = [
            {"ue_id": "imsi-208930000000002", "lat": 23.253045, "lng": 72.584939},
            {"ue_id": "imsi-208930000000001", "lat": 23.235491, "lng": 72.579252}
        ]
        gnb_locs_list = [
            {"id": 1, "lat": 23.249099, "lng": 72.579446, "radius": 1000},
            {"id": 3, "lat": 23.236352, "lng": 72.574773, "radius": 1000}
        ]
        zones = [{"zone_id": "airport-restricted", "zone_type": "RED"}]

    # 2. Convert lists to Dictionaries for O(1) lookup
    ue_lookup = {ue['ue_id']: ue for ue in ue_locs_list}
    
    # Crucial: Mapping Core ID "00000001" to Location ID "1"
    gnb_lookup = {f"{gnb['id']:08d}": gnb for gnb in gnb_locs_list}

    # 3. Build the "Agent-Ready" Structure
    processed_topology = {}

    for gnb in net_map:
        g_id = gnb['gnb_id']
        loc_info = gnb_lookup.get(g_id, {})
        
        # Get UEs specifically connected to THIS gNB
        active_ues = []
        for ue in gnb.get('connected_ues', []):
            supi = ue['supi']
            coords = ue_lookup.get(supi, {})
            active_ues.append({
                "supi": supi,
                "lat": coords.get("lat", "N/A"),
                "lng": coords.get("lng", "N/A")
            })

        processed_topology[f"gNodeB_{g_id}"] = {
            "lat": loc_info.get("lat", "N/A"),
            "lng": loc_info.get("lng", "N/A"),
            "radius": loc_info.get("radius", 0),
            "connected_ues": active_ues
        }

    final_output = {
        "network_state": processed_topology,
        "restricted_zones": zones
    }
    
    return final_output

In [14]:
data = get_processed_network_state()
# print(json.dumps(data, indent=3))

--- Starting Data Preprocessing Test ---

[Status] Successfully fetched live data from Localhost.


In [18]:
@tool
def get_flight_environment(query: str):
    """
    Returns the current 5G network topology, gNB coverage, 
    connected UEs, and Red/Restricted zones. 
    Use this to plan safe paths.
    """
    return get_processed_network_state()

# Initialize LLM
llm = ChatGoogleGenerativeAI(model=LLM_MODEL, temperature=0.1)

# System Prompt to handle the "Math Disclaimer"
sys_msg = """You are a UAV Path Orchestrator. 
Your task is to provide a safe list of waypoints from START to END.
CRITICAL RULES:
1. Avoid RED ZONES at all costs.
2. Stay within the radius of gNodeBs to ensure signal (C2 Link).
3. If a direct path is impossible, provide intermediate waypoints.
4. NOTE: As an LLM, your coordinate math is 'heuristic-based'. 
   State clearly that for production, a Graph-based Traversal (like A*) should validate your output.
"""

agent_executor = create_agent(
    llm, 
    tools=[get_flight_environment],
    system_prompt=sys_msg, # This is how we pass the system prompt now
    # response_format=""
)

In [20]:
import time
from tqdm import tqdm
from langchain_core.messages import HumanMessage

# Input coordinates
start_coords = "23.235, 72.579"
end_coords = "23.260, 72.605"

# Prepare the input in the required state format
inputs = {"messages": [HumanMessage(content=f"Plan a safe path from {start_coords} to {end_coords}.")]}

print(f"\n[Mission Start] Planning route for UAV...")

# We'll store the message here
final_answer = ""

# 1. Use stream_mode="values" to get the full state dictionary at each step
# 2. This avoids KeyError because the state always has the 'messages' key
with tqdm(total=5, desc="AI Reasoning Cycles") as pbar:
    for state in agent_executor.stream(inputs, stream_mode="values"):
        pbar.update(1)
        
        # In 'values' mode, 'state' is the actual state object
        # We grab the content of the very last message in the history
        last_msg = state["messages"][-1]
        
        # We only care about AI messages (the agent's response)
        if hasattr(last_msg, "content"):
            final_answer = last_msg.content
        
        time.sleep(0.4) # Simulated delay for the demo

print("\n" + "="*40)
print("     UAV PATH RECOMMENDATION ENGINE")
print("="*40)
print(final_answer)


[Mission Start] Planning route for UAV...


AI Reasoning Cycles:  60%|██████    | 3/5 [00:02<00:01,  1.19it/s]

--- Starting Data Preprocessing Test ---

[Status] Successfully fetched live data from Localhost.


AI Reasoning Cycles:  80%|████████  | 4/5 [00:06<00:01,  1.56s/it]


     UAV PATH RECOMMENDATION ENGINE
I have analyzed the flight environment and identified the following:

*   **gNodeBs:**
    *   gNodeB_00000001: (23.249099, 72.579446), Radius: 1000m
    *   gNodeB_00000002: (23.250344, 72.595505), Radius: 1000m
    *   gNodeB_00000003: (23.236352, 72.574773), Radius: 1000m
    *   gNodeB_00000004: (23.260947, 72.606442), Radius: 1000m
*   **Restricted Zones:**
    *   airport-restricted: Center (23.241357, 72.593231), Radius: 1000m (RED ZONE)

**Proposed Path:**

START: (23.235, 72.579)
Waypoint 1: (23.236, 72.585) - This waypoint is within the coverage of gNodeB_00000003 and gNodeB_00000001, and avoids the red zone.
Waypoint 2: (23.245, 72.590) - This waypoint is within the coverage of gNodeB_00000001 and gNodeB_00000002, and is close to the edge of the red zone, but still safe.
Waypoint 3: (23.255, 72.600) - This waypoint is within the coverage of gNodeB_00000002 and gNodeB_00000004, and is clear of any restricted zones.
END: (23.260, 72.605)

*